In [ ]:
import csv
import os
import time
import requests
import re
from datetime import datetime, timezone
from typing import Optional




GITHUB_PAT = ''

# Input and Output file names
INPUT_CSV_FILE = 'samsung_match.csv'
OUTPUT_CSV_FILE = '4Granular.csv'


API_CALL_DELAY_SECONDS = 1.2

 
# Granular 1
def github_repo_exists(repo_url: str) -> bool:
    try:
        match = re.search(r"github\.com/([^/]+)/([^/]+)", repo_url)
        if not match:
            return False
        owner = match.group(1)
        repo = match.group(2)
        base_repo_url = f"https://github.com/{owner}/{repo}"
        response = requests.head(base_repo_url, allow_redirects=True)
        return response.status_code == 200
    except requests.exceptions.RequestException:
        return False

# Granular 2
def check_version_and_get_sha(repo_url, version):
    match = re.search(r"github\.com/([^/]+)/([^/]+)", repo_url)
    if not match:
        print(f"--> Invalid GitHub URL: {repo_url}")
        return None, None, None
    owner, repo = match.groups()

    try:
        timestamp_sec = int(version) / 1000
        dt_object = datetime.fromtimestamp(timestamp_sec, tz=timezone.utc)
        iso_timestamp = dt_object.strftime('%Y-%m-%dT%H:%M:%SZ')
    except (ValueError, TypeError):
        print(f"--> Invalid timestamp format: {version}")
        return None, None, None

    api_url = f"https://api.github.com/repos/{owner}/{repo}/commits"
    headers = {'Authorization': f'token {GITHUB_PAT}', 'Accept': 'application/vnd.github.v3+json'}
    params = {'until': iso_timestamp, 'per_page': 1}
    
    time.sleep(API_CALL_DELAY_SECONDS)

    try:
        response = requests.get(api_url, headers=headers, params=params)
        if response.status_code in [404, 409, 403]: # Added 403 for PAT issues
            print(f"--> Repo not found, empty, private, or PAT invalid: {owner}/{repo} (Status: {response.status_code})")
            return None, None, None
        
        response.raise_for_status()
        commits = response.json()
        
        if commits:
            commit_sha = commits[0]['sha']
            print(f"--> Found version for timestamp {version} in {owner}/{repo}. Commit SHA: {commit_sha}")
            return commit_sha, owner, repo
        else:
            print(f"--> No version found for timestamp {version} in {owner}/{repo}")
            return None, None, None

    except requests.exceptions.RequestException as e:
        print(f"--> Request failed for {owner}/{repo}: {e}")
        return None, None, None

#Granular 3
def find_file_at_exact_path(full_file_url: str) -> bool:
    url_without_fragment = full_file_url.split('#')[0]
    try:
        response = requests.head(url_without_fragment, allow_redirects=True, timeout=10)
        return response.status_code == 200

    except requests.exceptions.RequestException:
        return False



#Granular 4
def find_method_in_file(url: str, method_name: str) -> Optional[str]:
    try:
        # 1. Convert URL to raw format and fetch content
        base_url = url.split('#')[0]
        raw_url = base_url.replace('github.com', 'raw.githubusercontent.com').replace('/blob/', '/').replace('/./', '/')

        
        response = requests.get(raw_url, timeout=10)
        response.raise_for_status()  # Check for HTTP errors (like 404 Not Found)

        # 2. Search for the method in the content
        for line in response.text.splitlines():
            if method_name in line:
                return line  

    except requests.exceptions.RequestException as e:
        # This block handles network, timeout, or HTTP errors
        print(f"Failed to process '{url}'. Reason: {e}")
        return None  # Return None on any exception

    # 3. If the loop completes, the method wasn't found
    return None


def main():
    print(f"Starting to process {INPUT_CSV_FILE}...")

    try:
        with open(INPUT_CSV_FILE, 'r', newline='', encoding='utf-8') as infile, \
             open(OUTPUT_CSV_FILE, 'w', newline='', encoding='utf-8') as outfile:

            reader = csv.reader(infile)
            writer = csv.writer(outfile)

            try:
                header = next(reader)
            except StopIteration:
                print("Error: Input CSV file is empty.")
                return

            # Add the new 'granular_level_reached' column to the header
            
            writer.writerow(header + ['granular_level_reached', 'raw_url', 'line_content', 'target_line_number'])


            # Get column indices from the header
            try:
                repo_url_idx = header.index('repository_url')
                version_idx = header.index('version')
                file_loc_idx = header.index('file_location')
                method_name_idx = header.index('method_name')
                function_code_idx = header.index('function_code') if 'function_code' in header else -1
            except ValueError as e:
                print(f"Error: Missing required column in CSV header: {e}")
                return

            # Process each row from the input file
            for i, row in enumerate(reader, 1):
                if not row: continue  # Skip empty rows

                print(f"\n--- Processing row {i} ---")

                # Skip rows that already have code and do NOT write them to the new file.
                if function_code_idx != -1 and row[function_code_idx].strip():
                    print(f"Skipping row {i} (function_code already present).")
                    continue

                # Extract data from the current row
                repo_url = row[repo_url_idx]
                version = row[version_idx]
                file_location_url = row[file_loc_idx]
                method_name = row[method_name_idx]
                
                # Use a single integer to track the level reached.
                granular_level_reached = 0
                raw_url = ""
                line_content = ""
                target_line_number = ""

                # --- Start Validation ---
                # Granular 1: Check if the repository exists
                if github_repo_exists(repo_url):
                    granular_level_reached = 1
                    print(f"Step 1: SUCCESS - Repository found at {repo_url}")
                    
                    # Granular 2: Check for a commit matching the version timestamp
                    commit_sha, _, _ = check_version_and_get_sha(repo_url, version)
                    if commit_sha:
                        granular_level_reached = 2
                        print(f"Step 2: SUCCESS - Version found with SHA: {commit_sha}")
                        
                        # Granular 3: Check if the file exists at the specified URL
                        if find_file_at_exact_path(repo_url):
                            granular_level_reached = 3
                            print(f"Step 3: SUCCESS - File found at path: {file_location_url}")
                            if find_method_in_file(repo_url, method_name):
                                granular_level_reached = 4
                                print(f"Step 5: SUCCESS - Method '{method_name}' found in the original file. Granular 5.")
                            else:
                                print(f"Step 3: FAILED - File not found at path: {file_location_url}")
                        else:
                            print(f"Step 2: FAILED - Version not found for timestamp.")
                    else:
                        print(f"Step 1: FAILED - Repository not found at {repo_url}")

                # Write the original row plus the final granular level reached
                writer.writerow(row + [granular_level_reached, raw_url, line_content, target_line_number])

    except FileNotFoundError:
        print(f"Error: The input file '{INPUT_CSV_FILE}' was not found. Please make sure it exists in the same directory.")
        return
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return

    print(f"\nProcessing complete. Output saved to {OUTPUT_CSV_FILE}")


if __name__ == '__main__':
    main()